# Train DistilBERT on a GPU (Kaggle or Colab)

Runs the repo's scripts on a GPU machine. Nothing is trained in the notebook itself, so the same commands work locally.

**Kaggle:** Settings → Accelerator → *GPU T4*, Internet → *On*. To use Kaggle's copy of the data, join the
[Toxic Comment Classification Challenge](https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge), accept its rules,
then *Add Input* → that competition. If it is not attached, the data is downloaded from a public mirror instead.

**Colab:** Runtime → Change runtime type → *T4 GPU*.

In [ ]:
REPO_URL = "https://github.com/shriniwas2412/iiith-mml-explainable-toxicity.git"

import os
if os.path.basename(os.getcwd()) != "repo":
    if not os.path.exists("repo"):
        !git clone -q {REPO_URL} repo
    %cd repo
!git pull -q
!pip install -q -e .

In [ ]:
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - turn the accelerator on")

## 1. Data and splits

In [ ]:
!python scripts/01_download_data.py
!python scripts/03_make_splits.py --subset 30000

## 2. Checkpoint 1: train on the 30k subset

A quick run to check everything works before spending GPU hours on the full data.

In [ ]:
!python scripts/05_train_transformer.py --train data/processed/train_subset_30000.csv --loss weighted_bce --out runs/distilbert_wbce_30k
!python scripts/06_tune_thresholds.py --run runs/distilbert_wbce_30k

## 3. Checkpoint 2: full training data, two loss functions

Weighted BCE against focal loss, same split and seed.

In [ ]:
!python scripts/05_train_transformer.py --loss weighted_bce --out runs/distilbert_wbce
!python scripts/06_tune_thresholds.py --run runs/distilbert_wbce

In [ ]:
!python scripts/05_train_transformer.py --loss focal --out runs/distilbert_focal
!python scripts/06_tune_thresholds.py --run runs/distilbert_focal

## 4. Identity-term bias audit

In [ ]:
!python scripts/07_fetch_bias_resources.py
!python scripts/08_build_templates.py
!python scripts/09_bias_audit.py --model hf:runs/distilbert_wbce/model --thresholds runs/distilbert_wbce/thresholds.json --out reports/bias/distilbert_wbce

## 5. Save the outputs

`results.zip` has the metric tables, thresholds, predictions and bias reports. `model.zip` has the fine-tuned weights for the demo.
On Kaggle both appear under *Output* after *Save Version*; on Colab download them from the file browser.

In [ ]:
!zip -qr results.zip runs reports -x "runs/*/model/*"
!zip -qr model.zip runs/distilbert_wbce/model
!ls -lh results.zip model.zip